# Loading a Hugging Face Model

This notebook demonstrates a clean Hugging Face model loading workflow using `transformers`.

Sections:
- Dependency installation
- Imports and environment setup
- Optional token configuration
- Model and tokenizer loading
- Text generation example

In [1]:
import sys
print(sys.executable)

!{sys.executable} -m pip install --upgrade pip
!{sys.executable} -m pip install \
    transformers==4.57.6 \
    huggingface_hub \
    python-dotenv

/Users/maazbaig/projects/llm_engineering/.venv/bin/python
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 38.6 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 25.0.1
    Uninstalling pip-25.0.1:
      Successfully uninstalled pip-25.0.1


In [2]:
import os
from dotenv import load_dotenv
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

load_dotenv()
print("Loaded environment variables.")

Loaded environment variables.


In [3]:
# Optional: load a Hugging Face token from .env
hf_token = os.getenv("HF_TOKEN")
if hf_token:
    print(f"HF token found and begins with: {hf_token[:6]}...")
else:
    print("No HF_TOKEN found; using public model access.")

HF token found and begins with: hf_mZA...


In [4]:
model_id = "meta-llama/Llama-3.2-3B-Instruct"
print(f"Model ID: {model_id}")

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16 if torch.backends.mps.is_available() else torch.float32,
    device_map="auto",
)
print("Model and tokenizer loaded successfully.")

Model ID: meta-llama/Llama-3.2-3B-Instruct


`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the disk.


Model and tokenizer loaded successfully.


In [8]:
prompt = "Explain QA testing in simple terms"

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=200,
    temperature=0.7
)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Explain QA testing in simple terms
**What is QA Testing?**

Imagine you're buying a new smartphone. Before you hand over your hard-earned cash, you want to make sure the phone works as promised. You'd test it out, see if it turns on, if the screen is responsive, if the camera takes good pictures, and so on.

**That's basically what QA testing is!**

QA (Quality Assurance) testing is the process of verifying that a product, like a software application, website, or even a physical product, works correctly and meets the requirements set by the developers.

**Why is QA testing important?**

In today's digital age, software applications are complex and interact with various systems, users, and devices. To ensure a smooth user experience, it's crucial to identify and fix bugs, errors, and inconsistencies before they cause problems for users.

**What does QA testing involve?**

Here are some common aspects of QA testing:

1. **Manual testing**: Human testers manually interact with the


In [ ]:
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    trust_remote_code=False,
)

prompt = "Write a short test case for a login page."
result = pipe(prompt, max_new_tokens=100)
print(result[0]["generated_text"])

In [7]:
from transformers import pipeline

pipe = pipeline(
    "text-generation",
    model="meta-llama/Llama-3.2-3B-Instruct"
)

print(pipe("Write a test case for login page", max_new_tokens=150))

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Device set to use mps:0
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


[{'generated_text': "Write a test case for login page, where a user logs in successfully and the login status is set to 1, which indicates successful login.\n\n```python\nimport unittest\nfrom your_module import LoginPage, LoginService\n\nclass TestLoginPage(unittest.TestCase):\n\n    def setUp(self):\n        self.login_service = LoginService()\n        self.login_page = LoginPage(self.login_service)\n\n    def test_login_success(self):\n        # Given\n        username = 'test_user'\n        password = 'test_password'\n\n        # When\n        self.login_page.login(username, password)\n\n        # Then\n        self.assertEqual(self.login_page.get_login_status(), 1)\n\nif __name__ == '__main__':\n    unittest.main()\n```\n\nIn the above code, we have written a test case for the login page"}]


## Diagnostics and Cleanup

Use this section to inspect the Hugging Face cache and release model resources.

In [ ]:
import os

cache_path = os.path.expanduser("~/.cache/huggingface/hub")
print("Cache path:", cache_path)
!ls {cache_path} | wc -l

In [5]:
ls ~/.cache/huggingface/hub | grep models--

models--meta-llama--Llama-3.2-3B-Instruct/


In [ ]:
# Clean up model resources and free MPS memory if available.
del pipe
if "model" in globals():
    del model
if "tokenizer" in globals():
    del tokenizer

if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    try:
        torch.mps.empty_cache()
        print("Cleared MPS cache.")
    except Exception as e:
        print("MPS cache clear failed:", e)

In [6]:
from huggingface_hub import scan_cache_dir

cache = scan_cache_dir()

print("Models found:")
for repo in cache.repos:
    print("-", repo.repo_id)

print("\nTotal models:", len(cache.repos))

Models found:
- meta-llama/Llama-3.2-3B-Instruct

Total models: 1


## Notes

- If you need private model access, set `HF_TOKEN` in a `.env` file.
- Adjust `model_id`, `max_new_tokens`, and other generation parameters as needed.